In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report

In [2]:
data = pd.read_csv('data.csv')
data

,process.b1.capacity,process.b2.capacity,process.b3.capacity,process.b4.capacity,property.price,property.product,property.winner,verification.result,verification.time
0,0,0,2,1,59,1,0,False,163.316667
1,0,0,2,1,59,2,0,False,200.860000
2,0,0,2,1,59,4,0,False,154.888889
3,0,0,2,1,59,6,0,False,108.640000
4,0,0,2,1,60,1,0,True,85.466667
...,...,...,...,...,...,...,...,...,...
2038,2,3,2,1,90,5,0,True,82.425000
2039,2,3,2,1,90,5,1,True,1316.983333
2040,2,3,2,1,90,5,2,False,9365.450000
2041,2,3,2,1,90,5,3,False,8474.025000


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2043 entries, 0 to 2042
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   process.b1.capacity  2043 non-null   int64  
 1   process.b2.capacity  2043 non-null   int64  
 2   process.b3.capacity  2043 non-null   int64  
 3   process.b4.capacity  2043 non-null   int64  
 4   property.price       2043 non-null   int64  
 5   property.product     2043 non-null   int64  
 6   property.winner      2043 non-null   int64  
 7   verification.result  2043 non-null   bool   
 8   verification.time    2043 non-null   float64
dtypes: bool(1), float64(1), int64(7)
memory usage: 129.8 KB


In [4]:
data['total_capacity'] = data['process.b1.capacity'] + data['process.b2.capacity'] + data['process.b3.capacity'] + data['process.b4.capacity']
data['price_per_capacity'] = data['property.price']/data['total_capacity'] # creating a new feature
data

,process.b1.capacity,process.b2.capacity,process.b3.capacity,process.b4.capacity,property.price,property.product,property.winner,verification.result,verification.time,total_capacity,price_per_capacity
0,0,0,2,1,59,1,0,False,163.316667,3,19.666667
1,0,0,2,1,59,2,0,False,200.860000,3,19.666667
2,0,0,2,1,59,4,0,False,154.888889,3,19.666667
3,0,0,2,1,59,6,0,False,108.640000,3,19.666667
4,0,0,2,1,60,1,0,True,85.466667,3,20.000000
...,...,...,...,...,...,...,...,...,...,...,...
2038,2,3,2,1,90,5,0,True,82.425000,8,11.250000
2039,2,3,2,1,90,5,1,True,1316.983333,8,11.250000
2040,2,3,2,1,90,5,2,False,9365.450000,8,11.250000
2041,2,3,2,1,90,5,3,False,8474.025000,8,11.250000


In [8]:
X = data.drop(columns=['total_capacity', 'verification.result', 'verification.time'])
y = data['verification.result']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
scaler = StandardScaler()
scaler.fit(x_train)
data_train = scaler.transform(x_train)
data_test = scaler.transform(x_test)

In [10]:
clf_1 = RandomForestClassifier(random_state=42)
clf_1.fit(data_train, y_train)
clf_1.score(data_train, y_train)

1.0

In [11]:
parameters_1 = {'n_estimators':[50, 100, 200], 'criterion': ['gini', 'entropy'], 'max_depth':[None, 10, 20, 30],
              'min_samples_split': [2, 5, 10, 20], 'min_samples_leaf': [1, 2, 4], 'max_features': ['sqrt', 'log2']}
clf_1_opt = GridSearchCV(clf_1, parameters_1, n_jobs=-1)
clf_1_opt.fit(data_train, y_train)
clf_1_opt.best_params_

{'criterion': 'gini',
 'max_depth': None,
 'max_features': 'log2',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 200}

In [12]:
final_model_1 = clf_1_opt.best_estimator_
final_model_1.score(data_train, y_train)

1.0

In [13]:
final_model_1.feature_importances_

array([0.04079385, 0.03802621, 0.00930415, 0.0163216 , 0.30226574,
       0.1454517 , 0.32749803, 0.12033872])

In [14]:
clf_2 = LGBMClassifier(verbose=-1)
clf_2.fit(data_train, y_train)
clf_2.score(data_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1.0

In [15]:
parameters_2 = {'num_leaves': [7, 5, 31], 'max_depth':[-1, 3, 5], 'learning_rate': [0.01, 0.1, 1], 'n_estimators': [50, 100, 200],
                'min_child_samples': [10, 20, 30]}
clf_2_opt = GridSearchCV(clf_2, parameters_2, n_jobs=-1)
clf_2_opt.fit(data_train, y_train)
clf_2_opt.best_params_

{'learning_rate': 1,
 'max_depth': 5,
 'min_child_samples': 30,
 'n_estimators': 50,
 'num_leaves': 31}

In [16]:
final_model_2 = clf_2_opt.best_estimator_
final_model_2.score(data_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1.0

In [17]:
estimators=[('rf', RandomForestClassifier(n_estimators=10, random_state=42)), ('knn', KNeighborsClassifier()), ('lr', LogisticRegression())]
clf_3 = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
clf_3.fit(data_train, y_train)
clf_3.score(data_train, y_train)

0.9981640146878825

In [18]:
parameters_3 = {'rf__n_estimators': [10, 20, 50], 'knn__n_neighbors':[3, 5, 7], 'lr__C': [0.01, 0.1, 1]}
clf_3_opt = GridSearchCV(clf_3, parameters_3, n_jobs=-1)
clf_3_opt.fit(data_train, y_train)
clf_3_opt.best_params_

{'knn__n_neighbors': 3, 'lr__C': 0.01, 'rf__n_estimators': 50}

In [19]:
final_model_3 = clf_3_opt.best_estimator_
final_model_3.score(data_train, y_train)

1.0

In [20]:
predict_1 = final_model_1.predict(data_test)
predict_2 = final_model_2.predict(data_test)
predict_3 = final_model_3.predict(data_test)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [21]:
print(final_model_1.score(data_test, y_test))
print(final_model_2.score(data_test, y_test))
print(final_model_3.score(data_test, y_test))

0.9731051344743277
1.0
0.9682151589242054


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:
print(classification_report(y_test, predict_1))

              precision    recall  f1-score   support

       False       0.98      0.99      0.98       354
        True       0.91      0.89      0.90        55

    accuracy                           0.97       409
   macro avg       0.95      0.94      0.94       409
weighted avg       0.97      0.97      0.97       409



In [23]:
print(classification_report(y_test, predict_2))

              precision    recall  f1-score   support

       False       1.00      1.00      1.00       354
        True       1.00      1.00      1.00        55

    accuracy                           1.00       409
   macro avg       1.00      1.00      1.00       409
weighted avg       1.00      1.00      1.00       409



In [24]:
print(classification_report(y_test, predict_3))

              precision    recall  f1-score   support

       False       0.98      0.98      0.98       354
        True       0.89      0.87      0.88        55

    accuracy                           0.97       409
   macro avg       0.93      0.93      0.93       409
weighted avg       0.97      0.97      0.97       409



As we can see, boosting showed the best result, but other algorithms were not much worse. It is easier for algorithms to predict negative result than the positive one.